# SVM phân loại mức độ nguy hiểm của động vật

Bài toán sử dụng các triệu chứng và tên động vật để dự đoán nhãn `Dangerous`. Vì các thuộc tính đầu vào là dạng phân loại, notebook mã hóa chúng bằng OneHotEncoder và huấn luyện SVM với kernel RBF.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay)

DATA_PATH = Path("../data/animal.csv")
df = pd.read_csv(DATA_PATH)
print(f"Kích thước dữ liệu: {df.shape}")
display(df.head())
print("\nKiểu dữ liệu và giá trị thiếu:")
display(pd.DataFrame({"kiểu dữ liệu": df.dtypes.astype(str), "giá trị thiếu": df.isna().sum()}))


## Khám phá nhãn và chuẩn bị dữ liệu

Các thuộc tính đầu vào được xử lý thiếu và mã hóa bên trong pipeline để tránh rò rỉ dữ liệu giữa tập huấn luyện và kiểm tra.

In [ ]:
TARGET = "Dangerous"
print("Phân bố nhãn:")
display(df[TARGET].value_counts(dropna=False).to_frame("số lượng"))
df_model = df.dropna(subset=[TARGET]).copy()
X = df_model.drop(columns=[TARGET]).astype("string")
y = df_model[TARGET].astype(str)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)


## Huấn luyện SVM

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
categorical_features = X.columns.tolist()
preprocessor = ColumnTransformer([("categorical", Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ("scale", StandardScaler(with_mean=False)),
]), categorical_features)])
model = Pipeline([("preprocess", preprocessor), ("classifier", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced"))])
model.fit(X_train, y_train)
print("Số mẫu huấn luyện:", len(X_train))


## Đánh giá mô hình

Dữ liệu kiểm tra được giữ riêng khỏi quá trình huấn luyện. Báo cáo precision, recall, F1 và ma trận nhầm lẫn cho từng lớp.

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", values_format="d")
plt.title("Ma trận nhầm lẫn - SVM")
plt.tight_layout()
plt.show()


## Nhận xét

Dựa trên classification report và ma trận nhầm lẫn ở trên, hãy nhận xét mô hình phân loại **Dangerous** tốt đến đâu, đặc biệt chú ý lớp ít xuất hiện và các trường hợp dự đoán sai.